# Drought Monitoring with Python & Google Earth Engine
## Turkana County, Kenya | Composite Drought Index (CDI)

This workshop notebook reviews the saved 2000–2025 Turkana raster products, then provides an optional live workflow for monthly rainfall, temperature, and vegetation monitoring. 

### Learning objectives
- Inspect and map multiband drought-index GeoTIFFs.
- Summarize annual county-wide index values and classify CDI severity.
- Fetch monthly ERA5-Land and MODIS data for Turkana with Google Earth Engine.
- Build a three-month, seasonally referenced CDI and an exploratory six-month forecast.

In [ ]:
%pip install earthengine-api geemap rasterio numpy pandas matplotlib statsmodels

In [ ]:
%pip install "drought-monitoring[all]"

## Indices and interpretation

The reference method combines precipitation (PDI), temperature (TDI), and vegetation (VDI):

$$CDI = 0.50 \times PDI + 0.25 \times TDI + 0.25 \times VDI$$

The supplied severity guide interprets CDI below 1 as drought, around 1 as near normal, and above 1 as wetter than normal. Classes are: `<0.50` extreme drought; `0.50–0.65` severe; `0.65–0.80` moderate; `0.80–0.90` mild; `0.90–1.10` near normal; `1.10–1.20` mild wet; `1.20–1.30` moderately wet; and `>1.30` very wet.

**Direction matters.** A warm-temperature anomaly increases drought stress, while a rainfall or vegetation deficit decreases those ratios. The live workflow therefore uses the inverse temperature ratio (`normal temperature / observed temperature`) so all three component scores point in the same direction: below 1 means drought stress. For clarity and stability, its simple three-month implementation uses seasonal ratios; the full run-length multiplier shown in the reference formula is not applied. The saved GeoTIFFs are reviewed exactly as supplied and are not recomputed here, so their CDI values may differ from the live, direction-aligned calculation.

In [ ]:
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import rasterio

plt.rcParams.update({"figure.figsize": (11, 5), "axes.grid": True, "grid.alpha": 0.25})
print("Core analysis libraries loaded.")

In [ ]:
from drought_monitoring.gee import authenticate

authenticate(project= "lunar-linker-509416-t1")

In [ ]:
ROOT = Path.cwd()
OUTPUT_DIR = ROOT / "Outputs"
START_YEAR = 2000
END_YEAR = 2025
WINDOW_MONTHS = 3
WEIGHTS = {"PDI": 0.50, "TDI": 0.25, "VDI": 0.25}

RASTER_PATHS = {
    name: OUTPUT_DIR / f"Turkana_2000_2025_{name}.tif"
    for name in ("CDI", "PDI", "TDI", "VDI")
}
missing_files = [str(path) for path in RASTER_PATHS.values() if not path.is_file()]
if missing_files:
    raise FileNotFoundError(f"Missing expected GeoTIFFs: {missing_files}")
print(f"Project folder: {ROOT}")
print(f"Saved Turkana products: {len(RASTER_PATHS)}")

In [ ]:
def years_from_raster(src, start_year=START_YEAR):
    descriptions = src.descriptions
    parsed_years = []
    for description in descriptions:
        try:
            parsed_years.append(int(str(description)[:4]))
        except (TypeError, ValueError):
            parsed_years = []
            break
    if len(parsed_years) == src.count:
        return parsed_years
    return list(range(start_year, start_year + src.count))

rasters = {}
raster_metadata = {}
for name, path in RASTER_PATHS.items():
    with rasterio.open(path) as src:
        rasters[name] = src.read(masked=True).filled(np.nan).astype("float32")
        raster_metadata[name] = {
            "years": years_from_raster(src),
            "crs": src.crs,
            "transform": src.transform,
            "bounds": src.bounds,
            "shape": (src.height, src.width),
            "bands": src.count,
        }

reference = raster_metadata["CDI"]
for name, metadata in raster_metadata.items():
    if (metadata["crs"] != reference["crs"] or
            metadata["transform"] != reference["transform"] or
            metadata["shape"] != reference["shape"] or
            metadata["years"] != reference["years"]):
        raise ValueError(f"{name} grid or years do not match the CDI raster.")

print("Product | CRS | rows x columns | bands | bounds")
for name, metadata in raster_metadata.items():
    print(name, metadata["crs"], metadata["shape"], metadata["bands"], metadata["bounds"])
years = reference["years"]
print(f"Year range: {years[0]}–{years[-1]} ({len(years)} bands)")

In [ ]:
def area_weighted_annual_mean(stack, transform):
    """Average each raster band with approximate cell-area weights for EPSG:4326."""
    row_latitudes = transform.f + (np.arange(stack.shape[1]) + 0.5) * transform.e
    row_weights = np.cos(np.deg2rad(row_latitudes))[:, None]
    valid = np.isfinite(stack)
    weighted_sum = np.where(valid, stack * row_weights[None, :, :], 0).sum(axis=(1, 2))
    weight_sum = np.where(valid, row_weights[None, :, :], 0).sum(axis=(1, 2))
    return np.divide(weighted_sum, weight_sum, out=np.full_like(weighted_sum, np.nan), where=weight_sum > 0)

annual_summary = pd.DataFrame(
    {name: area_weighted_annual_mean(stack, reference["transform"]) for name, stack in rasters.items()},
    index=pd.Index(years, name="year"),
)
print(annual_summary.round(3).to_string())

In [ ]:
fig, ax = plt.subplots(figsize=(12, 5))
for name, color in (("CDI", "#b33b32"), ("PDI", "#176b87"), ("TDI", "#d66a22"), ("VDI", "#278451")):
    ax.plot(annual_summary.index, annual_summary[name], marker="o", ms=3, lw=1.5, label=name, color=color)
ax.axhline(1, color="#333333", lw=1, ls="--", label="Reference value (1)")
ax.set(title="Annual area-weighted drought indices — Turkana, 2000–2025", xlabel="Year", ylabel="Index value")
ax.legend(ncol=5, frameon=False)
fig.tight_layout()
fig.savefig(OUTPUT_DIR / "Turkana_annual_indices_notebook.png", dpi=180, bbox_inches="tight")
plt.show()

In [ ]:
annual_cdi = annual_summary["CDI"].dropna()
fig, ax = plt.subplots(figsize=(12, 5))
ax.plot(annual_cdi.index, annual_cdi.values, color="#17324d", marker="o", lw=2)
ax.axhline(1.0, color="#222222", ls="--", lw=1.2, label="Near-normal reference")
for threshold in (0.50, 0.65, 0.80, 0.90, 1.10, 1.20, 1.30):
    ax.axhline(threshold, color="#8b8b8b", ls=":", lw=0.7)
ax.set(
    title="Annual Composite Drought Index — Turkana, 2000–2025",
    xlabel="Year",
    ylabel="Area-weighted CDI",
)
ax.set_xticks(annual_cdi.index[::2])
ax.legend(frameon=False)
fig.tight_layout()
fig.savefig(OUTPUT_DIR / "CDI_annual_Turkana.png", dpi=180, bbox_inches="tight")
plt.show()

In [ ]:
latest_position = int(np.argmax(years))
latest_year = years[latest_position]
bounds = reference["bounds"]
fig, ax = plt.subplots(figsize=(8, 7))
image = ax.imshow(
    rasters["CDI"][latest_position],
    extent=(bounds.left, bounds.right, bounds.bottom, bounds.top),
    origin="upper", cmap="RdYlBu", vmin=0.3, vmax=1.3, interpolation="nearest",
)
fig.colorbar(image, ax=ax, label="Saved CDI value (below 1 indicates drought in the supplied guide)")
ax.set(title=f"Saved CDI raster — Turkana, {latest_year}", xlabel="Longitude", ylabel="Latitude")
fig.tight_layout()
fig.savefig(OUTPUT_DIR / f"Turkana_CDI_{latest_year}_notebook.png", dpi=180, bbox_inches="tight")
plt.show()

In [ ]:
ncols = 5
nrows = int(np.ceil(len(years) / ncols))
fig, axes = plt.subplots(nrows, ncols, figsize=(16, 2.8 * nrows), squeeze=False)
extent = (bounds.left, bounds.right, bounds.bottom, bounds.top)

for ax, year, cdi_band in zip(axes.flat, years, rasters["CDI"]):
    image = ax.imshow(
        cdi_band, extent=extent, origin="upper", cmap="RdYlBu",
        vmin=0.3, vmax=1.3, interpolation="nearest",
    )
    ax.set_title(str(year))
    ax.set_xticks([])
    ax.set_yticks([])

for ax in axes.flat[len(years):]:
    ax.set_visible(False)

fig.colorbar(image, ax=axes.ravel().tolist(), shrink=0.82, label="Composite Drought Index (CDI)")
fig.suptitle("Annual CDI maps — Turkana, 2000–2025", y=1.01)
fig.savefig(OUTPUT_DIR / "CDI_annual_maps.png", dpi=180, bbox_inches="tight")
plt.show()

In [ ]:
SEVERITY_BINS = [-np.inf, 0.50, 0.65, 0.80, 0.90, 1.10, 1.20, 1.30, np.inf]
SEVERITY_LABELS = [
    "Extreme drought", "Severe drought", "Moderate drought", "Mild drought",
    "Near normal", "Mild wet", "Moderately wet", "Very wet",
]
annual_summary["CDI_class"] = pd.cut(
    annual_summary["CDI"], bins=SEVERITY_BINS, labels=SEVERITY_LABELS, right=False
)
display(annual_summary[["CDI", "CDI_class"]].round(3))
annual_summary.to_csv(OUTPUT_DIR / "Turkana_annual_indices_notebook.csv")

## live monthly analysis with Earth Engine

This section uses the GAUL level-1 Turkana boundary, ERA5-Land monthly precipitation and 2 m temperature, and MODIS MOD13A3 monthly NDVI. The date filter ends at `2026-01-01` because Earth Engine's end date is exclusive; this includes all months through December 2025. First-time setup may open a browser for authentication.

In [ ]:
import ee

GEE_PROJECT = "marine-infinity-470108-c6"  # Replace with your Google Cloud project ID if required by your account.
ee.Authenticate()
if GEE_PROJECT:
    ee.Initialize(project=GEE_PROJECT)
else:
    ee.Initialize()
print("Google Earth Engine initialized successfully.")

In [ ]:
import geemap

turkana = (
    ee.FeatureCollection("FAO/GAUL/2025/level1")
    .filter(ee.Filter.eq("GAUL0_NAME", "Kenya"))
    .filter(ee.Filter.eq("GAUL1_NAME", "Turkana"))
)
if turkana.size().getInfo() != 1:
    raise ValueError("Expected one Turkana boundary in FAO/GAUL/2025/level1. Check the dataset or region name.")
aoi = turkana.geometry()
print("AOI area (km²):", round(aoi.area(maxError=100).divide(1e6).getInfo()))

gee_map = geemap.Map(center=[3.2, 35.6], zoom=7)
gee_map.addLayer(turkana, {"color": "#f06b35"}, "Turkana County")
gee_map.addLayerControl()
gee_map

In [ ]:
def monthly_area_mean(collection_id, band, scale=10_000):
    """Reduce each monthly Earth Engine image to its Turkana-wide mean."""
    collection = (
        ee.ImageCollection(collection_id)
        .filterDate(f"{START_YEAR}-01-01", f"{END_YEAR + 1}-01-01")
        .filterBounds(aoi)
        .select(band)
    )

    def summarize(image):
        value = image.reduceRegion(
            reducer=ee.Reducer.mean(), geometry=aoi, scale=scale,
            maxPixels=1e10, bestEffort=True,
        ).get(band)
        return ee.Feature(None, {"month": image.date().format("YYYY-MM"), "value": value})

    features = collection.map(summarize).getInfo()["features"]
    records = [
        (feature["properties"].get("month"), feature["properties"].get("value"))
        for feature in features
        if feature["properties"].get("value") is not None
    ]
    if not records:
        raise ValueError(f"No values returned for {collection_id}/{band}. Check authentication and AOI.")
    dates = pd.to_datetime([f"{month}-01" for month, _ in records])
    values = [float(value) for _, value in records]
    return pd.Series(values, index=dates, dtype="float64").sort_index()

In [ ]:
monthly_dates = pd.date_range(f"{START_YEAR}-01-01", f"{END_YEAR}-12-01", freq="MS", name="month")
climate = pd.DataFrame(index=monthly_dates)
climate["precip_mm"] = monthly_area_mean(
    "ECMWF/ERA5_LAND/MONTHLY_AGGR", "total_precipitation_sum"
).mul(1000).reindex(monthly_dates)
climate["temp_c"] = monthly_area_mean(
    "ECMWF/ERA5_LAND/MONTHLY_AGGR", "temperature_2m"
).sub(273.15).reindex(monthly_dates)
climate["ndvi"] = monthly_area_mean(
    "MODIS/061/MOD13A3", "NDVI"
).mul(0.0001).reindex(monthly_dates)

print("Monthly rows expected/received:", len(monthly_dates), len(climate))
display(climate.isna().sum().rename("missing months").to_frame())
display(climate.describe().round(3))
climate.plot(subplots=True, figsize=(12, 8), title="Turkana monthly climate and vegetation inputs", legend=False)
plt.tight_layout()
plt.show()

## Calculate three-month drought scores

For each series, the interest period is a trailing three-month mean. Its baseline is the long-term mean for the same ending calendar month, which reduces seasonal bias. PDI and VDI are observed-to-normal ratios. TDI is normal-to-observed temperature, so unusually hot periods lower the score just as rainfall and vegetation deficits do. The weighted CDI then uses the reference 50/25/25 weights. This compact ratio implementation deliberately omits the separate run-length multiplier in the reference equation; the three-month rolling window provides a simple persistence measure.

In [ ]:
def seasonal_ratio(series, window=WINDOW_MONTHS, inverse=False):
    """Return a three-month ratio to the same-season climatology."""
    period_mean = series.rolling(window=window, min_periods=window).mean()
    baseline = period_mean.groupby(period_mean.index.month).transform("mean")
    if inverse:
        return baseline.div(period_mean.where(period_mean > 0))
    return period_mean.div(baseline.where(baseline > 0))

if not np.isclose(sum(WEIGHTS.values()), 1.0):
    raise ValueError("CDI weights must sum to 1.")

indices = pd.DataFrame(index=climate.index)
indices["PDI"] = seasonal_ratio(climate["precip_mm"])
indices["TDI"] = seasonal_ratio(climate["temp_c"], inverse=True)
indices["VDI"] = seasonal_ratio(climate["ndvi"])
indices["CDI"] = sum(indices[name] * weight for name, weight in WEIGHTS.items())
indices["CDI_class"] = pd.cut(
    indices["CDI"], bins=SEVERITY_BINS, labels=SEVERITY_LABELS, right=False
)
display(indices.tail(12).round(3))

In [ ]:
fig, ax = plt.subplots(figsize=(13, 5))
ax.axhspan(0, 0.5, color="#a91622", alpha=0.12, label="Extreme drought band")
ax.plot(indices.index, indices["CDI"], color="#172b3a", lw=0.9, label="Monthly CDI")
ax.plot(indices.index, indices["CDI"].rolling(12, min_periods=6).mean(), color="#e07a28", lw=2, label="12-month mean")
for threshold in (0.5, 0.65, 0.8, 0.9, 1.0, 1.1, 1.2, 1.3):
    ax.axhline(threshold, color="#777777", lw=0.6, ls=":")
ax.set(title="Monthly Composite Drought Index — Turkana, 2000–2025", xlabel="Month", ylabel="CDI")
ax.legend(frameon=False, ncol=2)
fig.tight_layout()
fig.savefig(OUTPUT_DIR / "Turkana_monthly_CDI_notebook.png", dpi=180, bbox_inches="tight")
fig.savefig(OUTPUT_DIR / "CDI_timeseries_Turkana.png", dpi=180, bbox_inches="tight")
plt.show()

In [ ]:
seasonal = indices.groupby(indices.index.month)[["PDI", "TDI", "VDI", "CDI"]].agg(["mean", "std"])
month_labels = ["Jan", "Feb", "Mar", "Apr", "May", "Jun", "Jul", "Aug", "Sep", "Oct", "Nov", "Dec"]
fig, axes = plt.subplots(1, 4, figsize=(15, 4), sharey=True)
for ax, name, color in zip(axes, ("PDI", "TDI", "VDI", "CDI"), ("#176b87", "#bd4b31", "#278451", "#75428a")):
    means = seasonal[(name, "mean")]
    spread = seasonal[(name, "std")].fillna(0)
    ax.fill_between(range(1, 13), means - spread, means + spread, color=color, alpha=0.16, label="±1 SD")
    ax.plot(range(1, 13), means, marker="o", color=color)
    ax.axhline(1, color="#666666", ls="--", lw=0.8)
    ax.set(title=name, xticks=range(1, 13), xticklabels=month_labels, xlabel="Month")
    ax.tick_params(axis="x", rotation=45)
axes[0].set_ylabel("Index value")
fig.suptitle("Seasonal drought-index cycle — Turkana, 2000–2025")
fig.tight_layout()
fig.savefig(OUTPUT_DIR / "Turkana_seasonal_indices_notebook.png", dpi=180, bbox_inches="tight")
plt.show()

In [ ]:
seasonal_data = indices[["CDI"]].dropna().copy()
month_to_season = {
    12: "DJF", 1: "DJF", 2: "DJF",
    3: "MAM", 4: "MAM", 5: "MAM",
    6: "JJA", 7: "JJA", 8: "JJA",
    9: "SON", 10: "SON", 11: "SON",
}
seasonal_data["season"] = seasonal_data.index.month.map(month_to_season)
seasonal_data["season_year"] = seasonal_data.index.year + (seasonal_data.index.month == 12)
seasonal_by_year = seasonal_data.groupby(["season_year", "season"])["CDI"].agg(["mean", "count"])
complete_seasons = seasonal_by_year[seasonal_by_year["count"] == 3].reset_index()
season_order = ["DJF", "MAM", "JJA", "SON"]
seasonal_summary = complete_seasons.groupby("season")["mean"].agg(["mean", "std"]).reindex(season_order)

fig, ax = plt.subplots(figsize=(9, 5))
ax.bar(
    season_order,
    seasonal_summary["mean"],
    yerr=seasonal_summary["std"].fillna(0),
    capsize=5,
    color=["#176b87", "#278451", "#db8b2b", "#bf4b3a"],
)
ax.axhline(1, color="#444444", ls="--", lw=1, label="Near-normal reference")
ax.set(
    title="Seasonal Composite Drought Index — Turkana",
    xlabel="Season (DJF, MAM, JJA, SON)",
    ylabel="Mean CDI across complete seasons",
)
ax.legend(frameon=False)
fig.tight_layout()
fig.savefig(OUTPUT_DIR / "CDI_seasonal_Turkana.png", dpi=180, bbox_inches="tight")
plt.show()

In [ ]:
from statsmodels.tsa.holtwinters import ExponentialSmoothing

FORECAST_MONTHS = 6
future_dates = pd.date_range(climate.index[-1] + pd.offsets.MonthBegin(1), periods=FORECAST_MONTHS, freq="MS")
raw_forecast = pd.DataFrame(index=future_dates)
for column in ("precip_mm", "temp_c", "ndvi"):
    training = climate[column].interpolate(method="time").ffill().bfill()
    model = ExponentialSmoothing(
        training, trend="add", seasonal="add", seasonal_periods=12,
        initialization_method="estimated",
    ).fit(optimized=True)
    raw_forecast[column] = model.forecast(FORECAST_MONTHS).to_numpy()
raw_forecast["precip_mm"] = raw_forecast["precip_mm"].clip(lower=0)
raw_forecast["ndvi"] = raw_forecast["ndvi"].clip(-1, 1)

extended_climate = pd.concat([climate, raw_forecast])
forecast_indices = pd.DataFrame(index=future_dates)
all_extended_indices = pd.DataFrame(index=extended_climate.index)
all_extended_indices["PDI"] = seasonal_ratio(extended_climate["precip_mm"])
all_extended_indices["TDI"] = seasonal_ratio(extended_climate["temp_c"], inverse=True)
all_extended_indices["VDI"] = seasonal_ratio(extended_climate["ndvi"])
all_extended_indices["CDI"] = sum(all_extended_indices[name] * weight for name, weight in WEIGHTS.items())
forecast_indices = all_extended_indices.loc[future_dates]
display(forecast_indices.round(3))

In [ ]:
fig, ax = plt.subplots(figsize=(12, 5))
recent = indices["CDI"].tail(36)
ax.plot(recent.index, recent, color="#172b3a", label="Observed CDI")
ax.plot(forecast_indices.index, forecast_indices["CDI"], color="#e07a28", marker="o", ls="--", label="Six-month exploratory forecast")
ax.axvline(climate.index[-1], color="#777777", ls=":", label="Forecast starts")
ax.axhline(1, color="#666666", lw=0.8)
ax.set(title="Exploratory six-month CDI outlook — Turkana", xlabel="Month", ylabel="CDI")
ax.legend(frameon=False)
fig.tight_layout()
fig.savefig(OUTPUT_DIR / "Turkana_CDI_forecast_notebook.png", dpi=180, bbox_inches="tight")
plt.show()

indices.to_csv(OUTPUT_DIR / "Turkana_monthly_indices_notebook.csv")
forecast_indices.to_csv(OUTPUT_DIR / "Turkana_CDI_forecast_notebook.csv")

## Interpretation 

- The saved rasters are coarse, multiband spatial products; a county-wide average can hide local drought hotspots.
- Raster-derived annual results are read as supplied. The live monthly CDI uses a direction-aligned temperature ratio and does not reproduce the saved raster calculation.
- The live data depend on Earth Engine availability, the selected county boundary, and valid credentials. Review missing-month counts before interpretation.
- The six-month Holt-Winters forecast is a simple exploratory baseline with no prediction interval. It is not an official drought warning or a guarantee.
- Figures and CSV tables are written to `Outputs/` with `_notebook` in their names; the source GeoTIFFs are never overwritten.
